# LLM Endpoint Healthcheck

Bu notebook LLM tarafinda eksik olan seyin kutuphane mi, config mi, network/TLS mi, yoksa model response'u mu oldugunu ayirmak icin hazirlandi.

Key'i hucreye yazma. Ayarlar su sirayla okunur: terminal env, repo kokundeki `.env`, `llm/.env.local`, `secret/secrets.yaml`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_candidates = [cwd, cwd.parent]
repo_root = next((p for p in repo_candidates if (p / "llm" / "llm_anomaly.py").exists()), cwd)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("repo_root=", repo_root)

## 1. Config okunuyor mu?

Bu hucre key'i basmaz. Sadece key'in yuklenip yuklenmedigini ve hangi endpoint/model kullanildigini gosterir.

In [ ]:
from llm.llm_anomaly import load_llm_settings, mask_url

settings = load_llm_settings()
safe_settings = {
    "base_url": mask_url(str(settings.get("base_url"))),
    "model": settings.get("model"),
    "timeout_seconds": settings.get("timeout_seconds"),
    "api_key_loaded": bool(settings.get("api_key")),
}
safe_settings

## 2. TCP ve TLS handshake testi

Burada HTTP request atmadan once host'a baglanti ve TLS handshake denenir. Onceki hatadaki `_ssl.c:987 handshake operation timed out` bu adimda yakalanir.

In [ ]:
from urllib.parse import urlparse
import socket
import ssl
import time
import traceback

parsed = urlparse(str(settings["base_url"]))
host = parsed.hostname
port = parsed.port or (443 if parsed.scheme == "https" else 80)
connect_timeout = min(int(settings.get("timeout_seconds") or 120), 60)

print({"host": host, "port": port, "scheme": parsed.scheme, "connect_timeout": connect_timeout})
start = time.time()
try:
    raw_sock = socket.create_connection((host, port), timeout=connect_timeout)
    print("TCP OK", round(time.time() - start, 2), "sn")
    if parsed.scheme == "https":
        context = ssl.create_default_context()
        tls_start = time.time()
        with context.wrap_socket(raw_sock, server_hostname=host) as tls_sock:
            cert = tls_sock.getpeercert()
            print("TLS OK", round(time.time() - tls_start, 2), "sn", "version=", tls_sock.version())
            print("cert_subject=", cert.get("subject", [])[:1])
    else:
        raw_sock.close()
except Exception as exc:
    print("TCP/TLS FAILED:", repr(exc))
    traceback.print_exc(limit=2)

## 3. `/models` endpoint testi

`/models` 404/405 donebilir; bu tek basina chat'in calismadigi anlamina gelmez. Ama TLS timeout, 401 veya proxy hatalarini ayirmaya yarar.

In [ ]:
import json
import urllib.error
import urllib.request

models_url = str(settings["base_url"]).rstrip("/") + "/models"
request = urllib.request.Request(
    models_url,
    headers={"Authorization": f"Bearer {settings['api_key']}"},
    method="GET",
)

start = time.time()
try:
    with urllib.request.urlopen(request, timeout=int(settings["timeout_seconds"])) as response:
        body = response.read().decode("utf-8", errors="replace")
    print("MODELS OK", {"status": response.status, "elapsed_sec": round(time.time() - start, 2)})
    print(body[:1000])
except urllib.error.HTTPError as exc:
    body = exc.read().decode("utf-8", errors="replace")
    print("MODELS HTTP ERROR", {"status": exc.code, "elapsed_sec": round(time.time() - start, 2)})
    print(body[:1000])
except Exception as exc:
    print("MODELS FAILED", {"elapsed_sec": round(time.time() - start, 2), "error": repr(exc)})
    traceback.print_exc(limit=2)

## 4. Repo'nun kendi OpenAI-compatible call testi

Bu test `run-oracle` icinde kullanilan ayni `call_openai_compatible_chat` fonksiyonunu cagirir. Basarili olursa kisa bir JSON donmelidir.

In [ ]:
from llm.llm_anomaly import call_openai_compatible_chat

messages = [
    {"role": "system", "content": "Only return valid JSON."},
    {"role": "user", "content": "Return exactly this JSON shape with a short Turkish value: {\"ok\": true, \"message\": \"...\"}"},
]

start = time.time()
try:
    result = call_openai_compatible_chat(messages, timeout_seconds=int(settings["timeout_seconds"]))
    print("CHAT OK", {"elapsed_sec": round(time.time() - start, 2)})
    print(json.dumps(result, ensure_ascii=False, indent=2))
except Exception as exc:
    print("CHAT FAILED", {"elapsed_sec": round(time.time() - start, 2), "error": repr(exc)})
    traceback.print_exc(limit=2)

## 5. LangChain ile opsiyonel test

Notebook veya eski script LangChain kullaniyorsa bu hucre ayni endpoint'i `ChatOpenAI` ile dener. Import yoksa sadece repo akisini kullanman yeterli.

In [ ]:
try:
    from langchain_openai import ChatOpenAI

    lc_llm = ChatOpenAI(
        base_url=str(settings["base_url"]).rstrip("/"),
        api_key=settings["api_key"],
        model=str(settings["model"]),
        temperature=0,
        timeout=int(settings["timeout_seconds"]),
    )
    start = time.time()
    response = lc_llm.invoke("Sadece JSON don: {\"ok\": true}")
    print("LANGCHAIN CHAT OK", {"elapsed_sec": round(time.time() - start, 2)})
    print(response.content)
except ImportError as exc:
    print("LANGCHAIN SKIPPED: langchain_openai import edilemedi", repr(exc))
except Exception as exc:
    print("LANGCHAIN CHAT FAILED", {"elapsed_sec": round(time.time() - start, 2), "error": repr(exc)})
    traceback.print_exc(limit=2)

## Sonuc nasil okunur?

- `api_key_loaded=False`: env veya `secret/secrets.yaml` LLM key okumuyor.
- `TCP/TLS FAILED` ve handshake timeout: kutuphane degil, endpoint/network/proxy/TLS/OpenShift route problemidir.
- `/models` 401: key/auth problemi.
- `/models` 404/405 ama chat OK: problem yok, endpoint `/models` desteklemiyor olabilir.
- `CHAT OK`: LLM endpoint repo akisi icin calisiyor.
- `CHAT FAILED` HTTP 400/422: endpoint OpenAI-compatible parametrelerden birini desteklemiyor olabilir; body'ye bak.